# Project 15 — BROKEN notebook (over-specification & raw loadings)

Seeded bugs centred on rotational non-identifiability. Run it, read the diagnostics, fix each bug. Clean reference: `notebook.ipynb`; answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate(); X = data['X']; t = data['truth']

### BUG 1 — over-specifying the number of factors (K=5 when truth is 2).

Three surplus factors are non-identified: they soak up rotational freedom and wreck mixing without improving fit.

In [ ]:
N, D = X.shape
K_WRONG = 5   # BUG 1: truth is K=2
with pm.Model() as model:
    W = pm.Normal('W', 0.0, 1.0, shape=(D, K_WRONG))
    mu = pm.Normal('mu', 0.0, 1.0, shape=D)
    sigma = pm.HalfNormal('sigma', 1.0)
    z = pm.Normal('z', 0.0, 1.0, shape=(N, K_WRONG))
    pm.Normal('X', mu=pm.math.dot(z, W.T)+mu, sigma=sigma, observed=X)
    # BUG 2: too few tune steps for a poorly-identified geometry
    idata = pm.sample(draws=400, tune=200, chains=2, random_seed=RNG,
                      progressbar=False)

### BUG 3 — interpreting a raw loading entry as if it were identified.

Reporting `W[0,0]` as 'the loading of channel 0 on factor 0' is meaningless: it changes under any rotation of the latent space.

In [ ]:
# BUG 3: this number is not identified and its R-hat is huge.
w00 = idata.posterior['W'].values.reshape(-1, D, K_WRONG)[:,0,0]
print('reported W[0,0] =', w00.mean(), '+/-', w00.std())
print(az.summary(idata, var_names=['W']).iloc[:5][['mean','r_hat','ess_bulk']])
fig, ax = plt.subplots(figsize=(6,3.2))
ax.hist(w00, bins=40, color='#C44E52'); ax.set_title('W[0,0]: multimodal, non-identified')
plt.tight_layout()